# AutoML PyCaret para Series de Tiempo (Proyecto Integrado)

Este notebook contiene la prueba del módulo `time_series` de PyCaret basándose en los resultados obtenidos en el Avance 5.
Se realizarán dos configuraciones de evaluación:
1. **Univariado** (Solo variable objetivo `casos_sarampion`)
2. **Multivariado** (Incluyendo la variable exógena `total_dosis_lag_3m`)


## 1. Configuración y Carga de Datos

In [1]:
import pandas as pd
from pathlib import Path
from pycaret.time_series import *

# ── Rutas de Datos ──
base_path = Path("../../..") / "data" / "processed"

# ── Dataset de Features Engineered ──
df_fe = pd.read_csv(base_path / "dataset_sarampion_features_engineered.csv")
df_fe["fecha"] = pd.to_datetime(df_fe["fecha"])
df_fe = df_fe.sort_values("fecha").reset_index(drop=True)

# ── Dataset de Vacunación ──
df_vacunas = pd.read_csv(base_path / "dataset_sarampion_vacunas_2010-2023.csv")
df_vacunas["fecha"] = pd.to_datetime(
    df_vacunas["anio"].astype(str) + "-" + df_vacunas["mes"].astype(str) + "-01"
)
df_vacunas = df_vacunas.sort_values("fecha").reset_index(drop=True)

df_fe["brote"] = (df_fe["casos_sarampion"] > 0).astype(int)

# ── Función de integración de vacunas con desfase ──
def integrar_vacunas_con_desfase(df_principal, df_vac, desfase_meses=0):
    df_vac_lag = df_vac.copy()
    df_vac_lag["fecha_join"] = df_vac_lag["fecha"] + pd.DateOffset(months=desfase_meses)
    col_name = f"total_dosis_lag_{desfase_meses}m"
    df_vac_lag = df_vac_lag[["fecha_join", "total_dosis"]].rename(
        columns={"total_dosis": col_name}
    )
    df_res = pd.merge(
        df_principal, df_vac_lag, left_on="fecha", right_on="fecha_join", how="left"
    )
    df_res.drop(columns=["fecha_join"], inplace=True)
    df_res.dropna(subset=[col_name], inplace=True)
    return df_res.reset_index(drop=True)

# ── Construir df_model ──
DESFASE_VACUNAS = 3
df_model = integrar_vacunas_con_desfase(df_fe, df_vacunas, desfase_meses=DESFASE_VACUNAS)

# Filtrar período (ejemplo 2010 a 2023 como en Avance 5)
TRAIN_START_YEAR = 2010
TEST_END_YEAR = 2023
df_model = df_model[
    (df_model["anio"] >= TRAIN_START_YEAR) & (df_model["anio"] <= TEST_END_YEAR)
].copy()

# En PyCaret, para series de tiempo a veces es útil setear el índice como el objeto de tiempo
df_model_ts = df_model[['fecha', 'casos_sarampion', 'total_dosis_lag_3m']].copy()

# Para univariado
df_univariado = df_model_ts[['fecha', 'casos_sarampion']]

# Para multivariado
df_multivariado = df_model_ts[['fecha', 'casos_sarampion', 'total_dosis_lag_3m']]

# Lista de modelos específicos a comparar solicitados
modelos_a_probar = [
    'gbr_cds_dt', 'ada_cds_dt', 'lightgbm_cds_dt', 'lasso_cds_dt',
    'llar_cds_dt', 'en_cds_dt', 'huber_cds_dt', 'rf_cds_dt',
    'br_cds_dt', 'et_cds_dt', 'lr_cds_dt', 'dt_cds_dt',
    'knn_cds_dt', 'ridge_cds_dt', 'omp_cds_dt', 'arima',
    'xgboost_cds_dt', 'auto_arima'
]


## 2. Pruebas Univariadas (Sin variables exógenas)

In [2]:
# Inicializar entorno univariado

setup(
    data=df_univariado,
    target='casos_sarampion',
    index='fecha',
    fh=3,               # Forecast Horizon (ajustar según tu test, e.g., 12 meses)
    fold=3,              # Número de folds para evaluación cruzada
    session_id=123
)

,Description,Value
0,session_id,123
1,Target,casos_sarampion
2,Approach,Univariate
3,Exogenous Variables,Not Present
4,Original data shape,"(165, 1)"
5,Transformed data shape,"(165, 1)"
6,Transformed train set shape,"(162, 1)"
7,Transformed test set shape,"(3, 1)"
8,Rows with missing values,0.0%
9,Fold Generator,ExpandingWindowSplitter


In [3]:
# Entrenar y comparar rendimiento los modelos específicos para Univariado
best_models_uni = compare_models(
    include=modelos_a_probar,
    sort='MAE'  # Métrica principal de clasificación, puedes cambiar a RMSE o SMAPE
)


,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
dt_cds_dt,Decision Tree w/ Cond. Deseasonalize & Detrending,0.0039,0.0010,0.0110,0.0119,49645150447833.7812,2.0000,0.0000,0.0367
et_cds_dt,Extra Trees w/ Cond. Deseasonalize & Detrending,0.0048,0.0012,0.0136,0.0144,61151147267402.3281,2.0000,0.0000,0.8667
xgboost_cds_dt,Extreme Gradient Boosting w/ Cond. Deseasonalize & Detrending,0.0050,0.0012,0.0141,0.0148,63598516799129.1016,2.0000,0.0000,0.1100
rf_cds_dt,Random Forest w/ Cond. Deseasonalize & Detrending,0.0051,0.0013,0.0143,0.0150,64463978757114.7812,2.0000,0.0000,0.7133
knn_cds_dt,K Neighbors w/ Cond. Deseasonalize & Detrending,0.0078,0.0019,0.0220,0.0225,99290300895666.4375,2.0000,0.0000,0.0567
gbr_cds_dt,Gradient Boosting w/ Cond. Deseasonalize & Detrending,0.0119,0.0028,0.0335,0.0339,150929402823821.4688,2.0000,0.0000,1.0733
huber_cds_dt,Huber w/ Cond. Deseasonalize & Detrending,0.1272,0.0305,0.3597,0.3639,1620150499820089.7500,2.0000,0.0000,0.7167
ada_cds_dt,AdaBoost w/ Cond. Deseasonalize & Detrending,0.3187,0.0762,0.9081,0.9125,4089527805787817.0000,2.0000,0.0000,0.7333
omp_cds_dt,Orthogonal Matching Pursuit w/ Cond. Deseasonalize & Detrending,0.5258,0.1275,1.4942,1.5248,6729491248308243.0000,2.0000,0.0000,0.0267
lr_cds_dt,Linear w/ Cond. Deseasonalize & Detrending,0.5258,0.1275,1.4942,1.5248,6729491248308243.0000,2.0000,0.0000,1.9800


In [4]:
# Ver el mejor modelo
print(best_models_uni)


BaseCdsDtForecaster(fe_target_rr=[WindowSummarizer(lag_feature={'lag': [1]},
                                                   n_jobs=1)],
                    regressor=DecisionTreeRegressor(random_state=123),
                    window_length=1)


In [5]:
# Plots de diagnóstico del mejor modelo
plot_model(best_models_uni, plot='forecast')


## 3. Pruebas Multivariadas (Con variable exógena: total_dosis_lag_3m)

In [6]:
# Inicializar entorno multivariado con exógenas

setup(
    data=df_multivariado,
    target='casos_sarampion',
    index='fecha',
    fh=12,               # Forecast Horizon
    fold=3,
    enforce_exogenous=True,
    session_id=123
)

,Description,Value
0,session_id,123
1,Target,casos_sarampion
2,Approach,Univariate
3,Exogenous Variables,Present
4,Original data shape,"(165, 2)"
5,Transformed data shape,"(165, 2)"
6,Transformed train set shape,"(153, 2)"
7,Transformed test set shape,"(12, 2)"
8,Rows with missing values,0.0%
9,Fold Generator,ExpandingWindowSplitter


In [7]:
# Entrenar y comparar rendimiento en caso Multivariado
best_models_multi = compare_models(
    include=modelos_a_probar,
    sort='MAE'
)


,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
arima,ARIMA,2.4755,1.0336,6.4156,11.5301,5410365691543193.0000,1.9837,-0.0952,0.0667
auto_arima,Auto ARIMA,2.4755,1.0336,6.4156,11.5301,5410365691543193.0000,1.9837,-0.0952,0.2133
gbr_cds_dt,Gradient Boosting w/ Cond. Deseasonalize & Detrending,2.7622,1.2777,7.2922,14.5772,8819885896554172.0000,1.9971,-0.1044,0.0800
rf_cds_dt,Random Forest w/ Cond. Deseasonalize & Detrending,2.8897,1.2863,7.6738,14.6580,10163047836339486.0000,2.0000,-0.1206,0.1833
lr_cds_dt,Linear w/ Cond. Deseasonalize & Detrending,2.9423,1.1608,7.8394,13.1027,10631238416573474.0000,2.0000,-0.1256,0.0533
ridge_cds_dt,Ridge w/ Cond. Deseasonalize & Detrending,2.9423,1.1608,7.8394,13.1027,10631323093061418.0000,2.0000,-0.1256,0.0800
en_cds_dt,Elastic Net w/ Cond. Deseasonalize & Detrending,2.9436,1.1610,7.8436,13.1053,10649172731115098.0000,2.0000,-0.1256,0.0567
lasso_cds_dt,Lasso w/ Cond. Deseasonalize & Detrending,2.9441,1.1610,7.8453,13.1063,10656305761162974.0000,2.0000,-0.1255,0.0533
llar_cds_dt,Lasso Least Angular Regressor w/ Cond. Deseasonalize & Detrending,2.9441,1.1610,7.8453,13.1063,10656305766371514.0000,2.0000,-0.1255,0.0533
br_cds_dt,Bayesian Ridge w/ Cond. Deseasonalize & Detrending,2.9522,1.1618,7.8698,13.1165,10752879693658178.0000,2.0000,-0.1254,0.0567


In [8]:
# Ver el mejor modelo
print(best_models_multi)


ARIMA()


In [9]:
# Plots de diagnóstico del mejor modelo
plot_model(best_models_multi, plot='forecast')


¿Están sirviendo los datos?
Los datos presentan un gran desafío de inflación de ceros estructural (el ~90% de las fechas observadas no tienen casos de sarampión). Esto produce efectos engañosos en las métricas de Auto ML y modelos manuales:

Ilusión de Univariado: 

PyCaret reporta un MAE extremadamente bajo (0.0110) en su modelo Univariado usando Árboles de Decisión (dt_cds_dt). Sin embargo, este es un efecto típo de modelos basados en árboles frente a datos con muchos ceros: el modelo aprende a predecir cero constante. Como la mayoría de los casos reales son cero, el error absoluto medio se vuelve minúsculo, pero el modelo falla por completo en predecir cuándo va a ocurrir un brote real.

Impacto de la Exógena: 

En el modelo multivariado las métricas de error (MAE: 6.41 y RMSE: 11.53) suben en comparación al modelo univariado de PyCaret. Esto significa que la variable exógena (total_dosis) aporta muy poca capacidad predictiva estadística. Al forzar al modelo (ARIMA en este caso) a darle un peso a dicha variable, empeora el "ajuste perfecto al cero" que tenía el modelo univariado.

¿Cuál modelo sale mejor?

Si miramos puramente la métrica, el AutoML Univariado (Decision Tree) es el ganador absoluto (RMSE 0.01 vs RMSE 14.3 del SARIMAX manual). Sin embargo, este modelo no es útil para vigilancia epidemiológica. Solo está prediciendo que "no habrá casos nunca" y estadísticamente acierta el 90% de las veces.

El modelo SARIMAX manual del Avance 5 convergió a predecir la media del entrenamiento, que es un acercamiento más honesto y estadísticamente robusto frente al ruido blanco de la serie temporal inflada por ceros.

Conclusión: 

Dado el conjunto de datos actual, aplicar AutoML tan potente como PyCaret no resuelve la falta de señal predictible que anticipa los brotes. Los datos están tan dominados por ausencias (ceros) que cualquier modelo intentará optimizarse devolviendo valores nulos. El acercamiento manual o estadístico puro (SARIMAX o Holt-Winters) documentado en el Avance 5 resulta el más transparente y adecuado hasta que se tenga un dataset más nutrido en el contexto epidemiológico.